# ZelusBench: Dimensionality

**Can the model maintain higher-dimensional state?**

Varies spatial dimension (2D vs 3D) while randomizing ALL background conditions
(chain depth, DAG structure, distractors, transforms, point types).

- Dimensions: 2D, 3D
- 15 seeds per dimension = 30 scenarios, ~90 queries

In [ ]:
import kaggle_benchmarks as kbench
import numpy as np
import random
import re
import time

from zelusbench.scenarios.config import ScenarioConfig
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.parser import parse_model_response
from zelusbench.evaluation.scorer import score_query


def score_response(response_text, scenario):
    query_dicts = [q.to_dict() for q in scenario.queries]
    parts = re.split(r'\[Query\s+q_\d+\]', response_text)
    if len(parts) > 1:
        parts = parts[1:]
    if len(parts) != len(query_dicts):
        parts = [response_text] * len(query_dicts)
    return [score_query(qd, parse_model_response(rp, qd))
            for qd, rp in zip(query_dicts, parts)]


DIMS = [2, 3]
SEEDS = 15
TOTAL = len(DIMS) * SEEDS

print(f"ZelusBench Dimensionality")
print(f"Dimensions: {DIMS} | Seeds: {SEEDS} | Total: {TOTAL} scenarios")

In [ ]:
@kbench.task(name="zelusbench_dimensionality")
def zelusbench_dimensionality(llm) -> tuple[float, float]:
    """Dimensionality: accuracy by spatial dimension (randomized background).
    Returns (mean_accuracy, std_dev)."""

    _start = time.time()
    print(f"Running {TOTAL} scenarios across {len(DIMS)} dimensions...")
    print("=" * 60)

    all_scores = []
    dim_scores = {}
    scenario_num = 0

    for dim in DIMS:
        dim_scores[dim] = []
        for seed in range(SEEDS):
            scenario_num += 1
            unique_seed = dim * 1000 + seed
            rng = random.Random(unique_seed)
            cfg = ScenarioConfig.randomize_except(rng, pinned={
                "dim": dim,
                "num_queries": 3,
                "seed": unique_seed,
            })
            gen = ScenarioGenerator(cfg)
            scenario = gen.generate(f"dim_{dim}d_s{seed}")

            response = llm.prompt(scenario.prompt)
            scores = score_response(response, scenario)
            all_scores.extend(scores)

            avg = float(np.mean([s.score for s in scores]))
            dim_scores[dim].append(avg)

            q_depths = [s.chain_depth for s in scores]
            tiers = [s.tier.name for s in scores]
            bg = f"dag={cfg.dag_structure.name} depth={cfg.min_chain_depth}-{cfg.max_chain_depth} dist={cfg.distractor_level.name} tx={cfg.transform_density.name}"
            print(f"  [{scenario_num}/{TOTAL}] dim={dim}D seed={seed} "
                  f"avg={avg:.2f} q_depths={q_depths} tiers={tiers} | {bg}")

        dim_avg = float(np.mean(dim_scores[dim]))
        print(f"  >> {dim}D mean: {dim_avg:.3f}")

    print("\n" + "=" * 60)
    print("RESULTS BY DIMENSION:")
    for dim in DIMS:
        avg = float(np.mean(dim_scores[dim]))
        print(f"  {dim}D  accuracy={avg:.3f}")
        kbench.assertions.assert_true(
            avg >= 0,
            expectation=f"{dim}D: model should produce valid answers (got {avg:.3f})"
        )

    overall = float(np.mean([s.score for s in all_scores]))
    std = float(np.std([s.score for s in all_scores]))
    elapsed = time.time() - _start

    print(f"\nOverall: {overall:.3f} +/- {std:.3f}")
    print(f"Total queries: {len(all_scores)}")
    print(f"Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

    return overall, std


zelusbench_dimensionality.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_dimensionality